# 🐖 CERDOS 23 — Evaluación COCO armonizada de YOLO26n-seg (3 semillas)

**Artículo 1:** *Benchmarking Deep Learning Models for Pig Instance Segmentation: A Multidimensional Reliability Assessment*.

**Recursos:** Google Colab **GPU T4 o L4**. **No entrena modelos ni modifica sus pesos.**

### Objetivos
1. Evaluar los checkpoints `best.pt` de las semillas **2026, 2027 y 2028** en las **mismas 373 imágenes test y 4.648 máscaras de referencia** del Notebook 09.
2. Aplicar **idéntico COCOeval para máscaras** (`IoU=0.50:0.95`, paso 0.05, hasta 100 detecciones; confianza de predicción `0.001`, NMS IoU `0.70`, `retina_masks=True`).
3. Reportar mAP de máscaras, AP@50, AP@75, valores por **fecha**, variación descriptiva entre semillas y errores de conteo a un **umbral fijo exploratorio** de confianza `0.25`.
4. Recuperar los resultados armonizados de YOLO11n-seg y Mask R-CNN del Notebook 09 **sin recalcularlos**. La comparación con una sola semilla de esas arquitecturas es descriptiva.
5. Guardar **Excel, JSON, figuras PNG/PDF, protocolo y ZIP** en las carpetas del proyecto, con verificación SHA-256 y posibilidad de reutilizar predicciones terminadas.

> **Precaución científica:** estas imágenes test ya fueron inspeccionadas durante los Notebooks 00–21. Por tanto, **NO** constituyen un nuevo test ciego ni validación externa. El checkpoint de referencia para comparaciones primarias se fija en **semilla 2026**, seleccionada por validación antes de leer el test; las tres semillas se reportan completas, sin elegir la ganadora por resultados de test. El conteo a 0.25 no está optimizado para YOLO26.

## 1. Comprobar la GPU y fijar dependencias

Activa **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** (o L4). No hace falta volver a entrenar el modelo. La versión de Ultralytics coincide con la utilizada en el Notebook 22.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Activa GPU T4 o L4 en Google Colab antes de ejecutar.'
print('✅ GPU:',torch.cuda.get_device_name(0))

In [ ]:
%pip -q install "ultralytics==8.4.182" "pycocotools>=2.0.8,<3" "pyyaml>=6,<7" "pandas>=2,<3" "openpyxl>=3.1" pillow matplotlib

## 2. Montar Drive y verificar que **los datos y el protocolo no cambiaron**

Se comprueba el SHA-256 del YAML y la lista de prueba contra el protocolo del Notebook 09. Además, se cotejan **IDs, nombres y dimensiones** de las imágenes con el JSON COCO de referencia original. No se regenerará Ground Truth desde cero.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from contextlib import redirect_stdout
import hashlib, json, io, sys, platform, time, zipfile, gc, math
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
import ultralytics
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as maskutils

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
assert ROOT.is_dir(), f'No se encuentra el proyecto: {ROOT}'
SPLIT=ROOT/'01_Datasets'/'FaroPigSeg'/'split_fecha_v1'
YAML=SPLIT/'data_fecha.yaml'
MANIFEST=SPLIT/'particion_fecha_manifest.json'
REPORTS=ROOT/'03_Resultados'/'05_Reportes'
METRICS=ROOT/'03_Resultados'/'03_Metricas'
FIGURES=ROOT/'03_Resultados'/'04_Figuras'
TRAIN=ROOT/'03_Resultados'/'06_Entrenamiento'/'YOLO26n_fecha_v1'
EXPORT=ROOT/'05_Exportaciones_ZIP'
BASE09=REPORTS/'CERDOS_09_COCO_Armonizado'
PROTOCOL09=BASE09/'CERDOS_09_Protocolo.json'
GT_FILE=BASE09/'CERDOS_09_GT_test_coco.json'
REPORT09=REPORTS/'CERDOS_09_Armonizacion_COCO.json'
OUTPUT=REPORTS/'CERDOS_23_YOLO26_COCO_Test'
for folder in (OUTPUT, METRICS, FIGURES, EXPORT): folder.mkdir(parents=True,exist_ok=True)

assert ultralytics.__version__=='8.4.182', ('Se esperaba Ultralytics 8.4.182',ultralytics.__version__)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()

def read_json(p):
    return json.loads(Path(p).read_text(encoding='utf-8'))

def save_json_atomic(p,obj):
    p=Path(p);tmp=p.with_suffix(p.suffix+'.writing')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')
    tmp.replace(p)

for p in (YAML,MANIFEST,PROTOCOL09,GT_FILE,REPORT09):
    if not p.is_file():raise FileNotFoundError(f'Falta el archivo anterior: {p}')
ref09=read_json(PROTOCOL09)
cf=yaml.safe_load(YAML.read_text(encoding='utf-8'))
TEST_TXT=Path(cf['test'])
assert TEST_TXT.is_file(),f'Falta lista test: {TEST_TXT}'
assert sha256(YAML)==ref09['yaml_sha256'],'YAML distinto al Notebook 09'
assert sha256(TEST_TXT)==ref09['test_file_sha256'],'La lista test cambió desde Notebook 09'
paths=[Path(s.strip()) for s in TEST_TXT.read_text(encoding='utf-8').splitlines() if s.strip()]
assert len(paths)==373 and len(set(map(str,paths)))==373,'Se esperan 373 imágenes únicas'
assert all(p.is_file() for p in paths),'Faltan imágenes de prueba en Drive'
manifest=read_json(MANIFEST)['asignaciones_fecha']
assert all(manifest.get(p.name[:10])=='test' for p in paths),'Fechas test incorrectas'
assert len({p.name[:10] for p in paths})==5,'Se esperaban 5 fechas de test'
gt_dict=read_json(GT_FILE)
assert len(gt_dict['images'])==373 and len(gt_dict['annotations'])==4648,'COCO GT incompleto'
assert sorted(im['id'] for im in gt_dict['images'])==list(range(1,374)),'IDs GT no consecutivos'
for iid,p in enumerate(paths,1):
    assert gt_dict['images'][iid-1]['file_name']==p.name,('Desalineación COCO/imagen',iid,p.name)
assert ref09['pipeline']=='CERDOS_09_COCOv1' and ref09['max_dets']==100
print('✅ Dataset verificado: 373 imágenes test, 4.648 instancias, 5 fechas')
print('✅ GT COCO y protocolos del Notebook 09 intactos')
print('✅ Runtime:',torch.__version__,'| Ultralytics:',ultralytics.__version__)

## 3. Verificación rigurosa de las tres semillas y del checkpoint escogido

La semilla 2026 será la referencia de comparación principal **porque fue la que presentó el mejor mAP de validación entre las tres**, no porque obtuviera mejor resultado en test. **Todos** los resultados del test de las tres semillas se conservarán.

In [ ]:
SEMILLAS=(2026,2027,2028)
SEMILLA_REFERENCIA=2026    # Fijada por validación; nunca cambiar tras observar test
UMBRAL_CONTEO_FIJO=0.25  # Solo análisis descriptivo, no seleccionado en test
PROTO_PARAMS={
    'imgsz':int(ref09['yolo_imgsz']),
    'conf':float(ref09['yolo_conf']),
    'iou':float(ref09['yolo_iou_nms']),
    'max_det':int(ref09['max_dets']),
    'retina_masks':bool(ref09['yolo_retina_masks']),
}
assert PROTO_PARAMS=={'imgsz':640,'conf':0.001,'iou':0.7,'max_det':100,'retina_masks':True}
model_files={}
seed_metadata={}
for seed in SEMILLAS:
    run=TRAIN/f'seed_{seed}'
    weight=run/'weights'/'best.pt'
    done=run/'CERDOS_22_ENTRENAMIENTO_FINALIZADO.json'
    for p in (weight,done):
        if not p.is_file():raise FileNotFoundError(f'Falta modelo o comprobante para seed {seed}: {p}')
    info=read_json(done)
    assert int(info['config']['seed'])==seed
    assert info['config']['model']=='yolo26n-seg.pt'
    frozen=info['config']['frozen_inputs']
    assert frozen['yaml_sha256']==sha256(YAML),'YAML no coincide con el entrenamiento'
    assert frozen['test_txt_sha256']==sha256(TEST_TXT),'Lista test no coincide con el entrenamiento'
    whash=sha256(weight)
    assert info['best_weights_sha256']==whash,('Los pesos cambiaron',seed)
    model_files[seed]=weight
    seed_metadata[seed]={
        'seed':seed,'best_pt':str(weight),'best_sha256':whash,
        'n_epocas_ejecutadas':info.get('actual_history_rows'),
        'fecha_entrenamiento_utc':info.get('finished_utc'),
        'tipo_evaluacion':'test_by_date_existing',
    }
    print(f'✅ Semilla {seed}: pesos SHA-256 {whash[:16]}...')
PROTO={
    'pipeline':'CERDOS_23_YOLO26_COCO_v1',
    'reference_notebook':'CERDOS_09_COCOv1',
    'test_images':373,'test_gt_instances':4648,'test_dates':5,
    'test_txt_sha256':sha256(TEST_TXT),
    'yaml_sha256':sha256(YAML),
    'gt_json_sha256':sha256(GT_FILE),
    'coeval':'pycocotools.COCOeval iouType=segm, IoU=0.50:0.95, maxDets=[1,10,100]',
    'prediction':PROTO_PARAMS,
    'mask_threshold':0.5,
    'count_conf_threshold':UMBRAL_CONTEO_FIJO,
    'seeds':[int(s) for s in SEMILLAS],
    'seed_for_descriptive_comparison':SEMILLA_REFERENCIA,
    'seed_selection_basis':'validation best mask mAP only, before test',
    'weights_sha256':{str(s):seed_metadata[s]['best_sha256'] for s in SEMILLAS},
}
PROTO_FILE=OUTPUT/'CERDOS_23_Protocolo.json'
if PROTO_FILE.exists():
    assert read_json(PROTO_FILE)==PROTO,'Existen salidas de un protocolo distinto: no reutilizar sin auditar'
else:
    if any(OUTPUT.glob('CERDOS_23_Pred_seed_*.json')):
        raise RuntimeError('Se encontraron predicciones sin protocolo: revisar carpeta antes de continuar')
    save_json_atomic(PROTO_FILE,PROTO)
print('✅ Protocolo congelado sin necesidad de modificar los tres checkpoints')

## 4. Inferencia de las tres semillas (sin reentrenamiento)

Se crea una predicción COCO distinta para cada `best.pt`, usando los mismos 373 IDs del Notebook 09. Cada resultado completo se guarda con checksum y podrá reutilizarse si se interrumpe Google Colab. **Nunca** se cargan modelos del conjunto de prueba para realizar ajustes.

In [ ]:
def mask_to_rle(m):
    item=maskutils.encode(np.asfortranarray(m.astype(np.uint8)))
    item['counts']=item['counts'].decode('ascii')
    return item

predictions={}
execution_logs=[]
for seed in SEMILLAS:
    pred_path=OUTPUT/f'CERDOS_23_Pred_seed_{seed}.json'
    lock_path=OUTPUT/f'CERDOS_23_Pred_seed_{seed}_hash.json'
    if pred_path.is_file():
        if not lock_path.is_file():
            raise RuntimeError(f'Predicciones {seed} sin registro hash. No reutilizar datos sin auditar.')
        meta=read_json(lock_path)
        assert meta['protocolo_sha256']==sha256(PROTO_FILE)
        assert meta['weights_sha256']==seed_metadata[seed]['best_sha256']
        assert meta['predictions_sha256']==sha256(pred_path),f'Predicciones alteradas para {seed}'
        rows=read_json(pred_path)
        assert all(1<=int(z['image_id'])<=373 and z['category_id']==1 for z in rows)
        print(f'♻️ Semilla {seed}: {len(rows)} predicciones reutilizadas y verificadas.')
        execution_logs.append({'seed':seed,'reutilizado':True,'tiempo_s':None,'n_predicciones':len(rows)})
    else:
        model=YOLO(str(model_files[seed]))
        assert model.task=='segment',(seed,model.task)
        rows=[]
        start=time.perf_counter()
        for iid,p in enumerate(paths,1):
            result=model.predict(source=str(p),device=0,
                  imgsz=PROTO_PARAMS['imgsz'],conf=PROTO_PARAMS['conf'],
                  iou=PROTO_PARAMS['iou'],max_det=PROTO_PARAMS['max_det'],
                  retina_masks=PROTO_PARAMS['retina_masks'],verbose=False,save=False)[0]
            info=gt_dict['images'][iid-1]
            ow,oh=int(info['width']),int(info['height'])
            scores=(result.boxes.conf.detach().cpu().numpy()
                    if result.boxes is not None else np.empty(0))
            if len(scores):
                assert result.masks is not None and result.masks.data is not None,('No hay máscaras',seed,p.name)
                arr=result.masks.data.detach().cpu().numpy()
                assert arr.shape==(len(scores),oh,ow), ('Resolución inesperada, sin redimensionar máscaras a ciegas',seed,p.name,arr.shape,(len(scores),oh,ow))
                for score,m in zip(scores,arr):
                    bm=(m>0.5).astype(np.uint8)
                    if bm.any():
                        rows.append({'image_id':int(iid),'category_id':1,
                            'segmentation':mask_to_rle(bm),'score':float(score)})
            if iid%50==0:print(f'Semilla {seed}: {iid}/373 imágenes')
        duration=round(time.perf_counter()-start,1)
        save_json_atomic(pred_path,rows)
        save_json_atomic(lock_path,{
            'seed':seed,'weights_sha256':seed_metadata[seed]['best_sha256'],
            'protocolo_sha256':sha256(PROTO_FILE),
            'predictions_sha256':sha256(pred_path),
            'n_predictions':len(rows),'saved_utc':datetime.now(timezone.utc).isoformat(),
        })
        print(f'✅ Semilla {seed}: {len(rows)} predicciones COCO en {duration:.1f}s')
        execution_logs.append({'seed':seed,'reutilizado':False,'tiempo_s':duration,'n_predicciones':len(rows)})
        del model;torch.cuda.empty_cache();gc.collect()
    predictions[seed]=rows
print('✅ Predicciones archivadas de las tres semillas.')

## 5. Evaluación armonizada COCO: global y por fecha

**Métrica principal:** `mAP@50:95` de segmentación de instancias. Se usan las máscaras originales y `pycocotools.COCOeval('segm')`. Se informa también `mAP@50` y `mAP@75`. En cada fecha COCOeval recalcula las curvas dentro del subconjunto de imágenes; **no** se promedian arbitrariamente los AP de fechas para obtener el AP global.

In [ ]:
def groundtruth_coco(d):
    gt=COCO();gt.dataset=d
    with redirect_stdout(io.StringIO()):gt.createIndex()
    return gt

def pred_coco(gt,rows):
    # loadRes muta algunas estructuras de entrada: normalizar dict nuevo en cada evaluación.
    if rows:
        safe=[{'image_id':int(r['image_id']),'category_id':int(r['category_id']),
               'segmentation':r['segmentation'],'score':float(r['score'])} for r in rows]
        with redirect_stdout(io.StringIO()):dt=gt.loadRes(safe)
        return dt
    dt=COCO();dt.dataset={'images':gt.dataset['images'],'categories':gt.dataset['categories'],'annotations':[]}
    with redirect_stdout(io.StringIO()):dt.createIndex()
    return dt

def ap_segm(gt,rows,im_ids=None):
    dt=pred_coco(gt,rows)
    evaluator=COCOeval(gt,dt,'segm')
    evaluator.params.maxDets=[1,10,100]
    if im_ids is not None:
        evaluator.params.imgIds=sorted(int(x) for x in im_ids)
    with redirect_stdout(io.StringIO()):
        evaluator.evaluate();evaluator.accumulate();evaluator.summarize()
    return {'mAP50_95_mascaras':float(evaluator.stats[0]),
            'mAP50_mascaras':float(evaluator.stats[1]),
            'mAP75_mascaras':float(evaluator.stats[2])}

gt=groundtruth_coco(gt_dict)
bydate={}
for iid,p in enumerate(paths,1):bydate.setdefault(p.name[:10],[]).append(iid)
assert len(bydate)==5
metrics_global=[];metrics_date=[]
for seed in SEMILLAS:
    g=ap_segm(gt,predictions[seed])
    metrics_global.append({'modelo':'YOLO26n-seg','semilla':seed,'n_imagenes':373,
                           'gt_instancias':4648,'n_predicciones':len(predictions[seed]),**g})
    for d,ids in sorted(bydate.items()):
        item=ap_segm(gt,predictions[seed],ids)
        metrics_date.append({'modelo':'YOLO26n-seg','semilla':seed,'fecha':d,'n_imagenes':len(ids),**item})
    print(f'✅ YOLO26n-seg seed={seed}: mAP50:95={g["mAP50_95_mascaras"]:.4f}, AP50={g["mAP50_mascaras"]:.4f}, AP75={g["mAP75_mascaras"]:.4f}')
df_global=pd.DataFrame(metrics_global)
df_date=pd.DataFrame(metrics_date)
display(df_global)

# Estabilidad descriptiva de semillas: 3 valores NO bastan para inferencia robusta.
cols=['mAP50_95_mascaras','mAP50_mascaras','mAP75_mascaras']
seed_summary=[]
for col in cols:
    vals=df_global[col].astype(float)
    seed_summary.append({'metrica':col,'n_semillas':len(vals),'media':float(vals.mean()),
                         'desv_std_muestral':float(vals.std(ddof=1)),
                         'min':float(vals.min()),'max':float(vals.max()),
                         'amplitud':float(vals.max()-vals.min())})
df_summary=pd.DataFrame(seed_summary)
display(df_summary)

## 6. Conteo descriptivo por imagen — umbral fijo, **sin optimización en test**

Se cuentan las predicciones con confianza `≥ 0.25`. Este umbral sirve solo para caracterizar errores, **no** para declarar cuál modelo cuenta mejor ni para escoger el umbral óptimo de YOLO26. Una futura calibración y ajuste de umbral debe usar exclusivamente las 196 imágenes de **validación**.

In [ ]:
from collections import Counter
n_gt=Counter(int(a['image_id']) for a in gt_dict['annotations'])
per_image=[]
for seed in SEMILLAS:
    n_pred=Counter(int(r['image_id']) for r in predictions[seed]
                   if float(r['score'])>=UMBRAL_CONTEO_FIJO)
    for iid,p in enumerate(paths,1):
        true=int(n_gt.get(iid,0));pred=int(n_pred.get(iid,0));diff=pred-true
        per_image.append({'modelo':'YOLO26n-seg','semilla':seed,'image_id':iid,
                          'fecha':p.name[:10],'imagen':p.name,
                          'gt_conteo':true,'pred_conteo_025':pred,
                          'error_firmado_pred_menos_gt':diff,
                          'error_absoluto':abs(diff),
                          'error_relativo_pct':100*abs(diff)/true if true>0 else np.nan,
                          'conteo_exacto':int(diff==0),
                          'umbral_fijo':UMBRAL_CONTEO_FIJO})
df_image=pd.DataFrame(per_image)
assert len(df_image)==1119,('Se esperaban 373*3 registros',len(df_image))
count_global=df_image.groupby('semilla',as_index=False).agg(
    MAE_conteo_025=('error_absoluto','mean'),
    sesgo_promedio=('error_firmado_pred_menos_gt','mean'),
    error_relativo_medio_pct=('error_relativo_pct','mean'),
    pct_conteo_exacto=('conteo_exacto',lambda z:100*float(z.mean())),
    n_imagenes=('image_id','nunique'))
count_date=df_image.groupby(['semilla','fecha'],as_index=False).agg(
    MAE_conteo_025=('error_absoluto','mean'),
    sesgo_promedio=('error_firmado_pred_menos_gt','mean'),
    pct_conteo_exacto=('conteo_exacto',lambda z:100*float(z.mean())),
    n_imagenes=('image_id','nunique'))
print('✅ Conteo descriptivo a confianza fija 0,25 — NO optimizado en test:')
display(count_global)

## 7. Comparación descriptiva con los dos modelos anteriores

Se recuperan los datos del **Notebook 09** tal como fueron calculados originalmente; el Notebook 23 no los sobreescribe. La comparación de YOLO11n-seg y Mask R-CNN (una semilla cada uno) frente a YOLO26 (tres semillas) **no es todavía una prueba formal de superioridad entre arquitecturas**. El resultado de YOLO26 elegido para la tabla principal es la semilla **2026, preseleccionada en validación**.

In [ ]:
report09=read_json(REPORT09)
assert report09['protocolo']==ref09,'El informe 09 tiene distinto protocolo'
old=pd.DataFrame(report09['global']).copy()
assert {'YOLO11n-seg','Mask R-CNN ResNet50-FPN'}==set(old['modelo'])
old['semilla']='original'
old['fuente']='Notebook 09 — una ejecución por arquitectura'
new=df_global[df_global['semilla']==SEMILLA_REFERENCIA].copy()
new['fuente']='Notebook 23 — semilla preseleccionada por validación'
comparison=pd.concat([old,new],ignore_index=True,sort=False)
fields=['modelo','semilla','mAP50_95_mascaras','mAP50_mascaras','mAP75_mascaras','fuente']
df_comparison=comparison[fields].copy()
display(df_comparison)
print('⚠️ Comparación descriptiva: faltan semillas adicionales en YOLO11 y Mask R-CNN.')

## 8. Figuras profesionales para el artículo (en inglés)

Se guardan PNG a 450 dpi y PDF vectoriales. Los colores y los ejes son consistentes. La figura principal compara los modelos, mientras otra muestra las cinco fechas y las tres semillas.

In [ ]:
plt.rcParams.update({
    'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':12,
    'axes.labelsize':11,'xtick.labelsize':10,'ytick.labelsize':10,
    'figure.dpi':110,'savefig.dpi':450,'axes.spines.top':False,
    'axes.spines.right':False
})
C={'YOLO11n-seg':'#447AA3','Mask R-CNN ResNet50-FPN':'#C78062','YOLO26n-seg':'#2B927A'}

fig,ax=plt.subplots(figsize=(9.1,5.3))
order=['YOLO11n-seg','Mask R-CNN ResNet50-FPN','YOLO26n-seg']
vals=[float(df_comparison.set_index('modelo').loc[k,'mAP50_95_mascaras']) for k in order]
labels=['YOLO11n-seg','Mask R-CNN','YOLO26n-seg']
x=np.arange(3)
ax.scatter(x,vals,s=125,c=[C[k] for k in order],zorder=5)
for i,val in enumerate(vals):ax.annotate(f'{val:.3f}',(i,val),xytext=(0,10),textcoords='offset points',ha='center',fontweight='bold')
ax.set_xticks(x,labels)
ax.set_ylim(max(0,min(vals)-0.07),min(1,max(vals)+0.07))
ax.set_ylabel('Mask AP@[0.50:0.95] (COCO)')
ax.set_title('Pig instance segmentation: harmonized mask AP on 373 test images',pad=14,fontweight='bold')
ax.grid(axis='y',alpha=0.19)
ax.text(0.5,-0.19,'YOLO26: validation-selected seed 2026; YOLO11 and Mask R-CNN: original single runs.\nExploratory within FaroPigSeg; no external validation or equal-budget claim.',
        transform=ax.transAxes,ha='center',va='top',fontsize=8.9,color='#434B55')
fig.tight_layout(rect=[0,0.09,1,1])
fig.savefig(FIGURES/'CERDOS_23_AP_Global_3_Modelos.png',bbox_inches='tight')
fig.savefig(FIGURES/'CERDOS_23_AP_Global_3_Modelos.pdf',bbox_inches='tight')
plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(10.5,5.7))
dates=sorted(df_date['fecha'].unique())
colors=['#337AAB','#239C82','#A462AD']
for k,seed in enumerate(SEMILLAS):
    temp=df_date[df_date['semilla']==seed].set_index('fecha').loc[dates]
    ax.plot(np.arange(len(dates)),temp['mAP50_95_mascaras'].values,
            marker='o',linewidth=2.0,markersize=6.5,color=colors[k],label=f'Seed {seed}')
ax.set_xticks(np.arange(len(dates)),dates,rotation=25,ha='right')
ax.set_ylabel('Mask AP@[0.50:0.95] (COCO)')
ax.set_title('YOLO26n-seg: test-set performance across capture dates and training seeds',pad=12,fontweight='bold')
ax.set_ylim(0,1.02);ax.grid(axis='y',alpha=.22)
ax.legend(frameon=False,ncol=3,loc='lower center')
fig.tight_layout()
fig.savefig(FIGURES/'CERDOS_23_AP_Por_Fecha_Semillas.png',bbox_inches='tight')
fig.savefig(FIGURES/'CERDOS_23_AP_Por_Fecha_Semillas.pdf',bbox_inches='tight')
plt.show();plt.close(fig)

fig,ax=plt.subplots(figsize=(8.5,4.8))
xx=np.arange(len(SEMILLAS))
cbar=ax.bar(xx,count_global.set_index('semilla').loc[list(SEMILLAS),'MAE_conteo_025'],
            color=['#5486A8','#4D9C87','#9F8EB8'],width=.53)
ax.bar_label(cbar,fmt='%.2f',padding=5)
ax.set_xticks(xx,[str(s) for s in SEMILLAS]);ax.set_xlabel('Training seed')
ax.set_ylabel('Count MAE (pigs / image), score ≥ 0.25')
ax.set_title('YOLO26n-seg: exploratory counting at a fixed confidence threshold',pad=12,fontweight='bold')
ax.grid(axis='y',alpha=.18)
fig.tight_layout()
fig.savefig(FIGURES/'CERDOS_23_Conteo_Umbral_Fijo.png',bbox_inches='tight')
fig.savefig(FIGURES/'CERDOS_23_Conteo_Umbral_Fijo.pdf',bbox_inches='tight')
plt.show();plt.close(fig)
print('✅ Tres figuras PNG y PDF guardadas en 03_Resultados/04_Figuras.')

## 9. Excel completo, informe de reproducibilidad y ZIP

El archivo Excel contiene los resultados globales, por fecha, por imagen, resumen de las tres semillas, comparaciones y limitaciones. Se adjuntan también las **predicciones COCO de cada semilla** (RLE, sin imágenes originales ni checkpoints pesados). Los resultados quedan en Drive y se descargará automáticamente un ZIP para la auditoría.

In [ ]:
EXCEL=METRICS/'CERDOS_23_YOLO26_Evaluacion_COCO_3_Semillas.xlsx'
REPORT_FILE=OUTPUT/'CERDOS_23_Informe_Evaluacion.json'
REPORT={
    'etapa':'23_YOLO26_COCO_test_sin_entrenamiento',
    'fecha_utc':datetime.now(timezone.utc).isoformat(),
    'protocolo':PROTO,
    'gpu':torch.cuda.get_device_name(0),
    'torch':torch.__version__,
    'ultralytics':ultralytics.__version__,
    'pycocotools':'COCOeval segm',
    'modelos':seed_metadata,
    'global':metrics_global,
    'por_fecha':metrics_date,
    'resumen_semillas':seed_summary,
    'conteo_umbral_fijo':count_global.to_dict(orient='records'),
    'comparacion_descriptiva':df_comparison.to_dict(orient='records'),
    'inferencia':execution_logs,
    'limitaciones':[
        'Test FaroPigSeg ya inspeccionado en notebooks previos: evaluación ampliada posterior, NO validación externa.',
        'Tres semillas YOLO26 en test; YOLO11 y Mask R-CNN todavía con una ejecución cada uno.',
        'COCO AP global calculado sobre las 373 imágenes; AP por fecha no se promedia para recuperar AP global.',
        'Comparación de arquitecturas sin demostrar presupuesto de entrenamiento comparable.',
        'La exactitud del conteo a 0.25 es descriptiva; umbral NO optimizado en test ni calibrado.',
        'Las fechas o imágenes no demuestran independencia entre animales ni granjas.',
        'No se calcularon p-valores; n=3 semillas no fundamenta inferencia estadística robusta.',
        'No comparar latencias entre GPUs L4 y T4 obtenidas en sesiones diferentes.',
    ]
}
with pd.ExcelWriter(EXCEL,engine='openpyxl') as writer:
    for frame,name in [
        (df_global,'COCO_Global'),(df_date,'COCO_Por_Fecha'),
        (df_summary,'Estabilidad_Semillas'),(df_image,'Conteo_Por_Imagen'),
        (count_global,'Conteo_Global'),(count_date,'Conteo_Por_Fecha'),
        (df_comparison,'Tres_Arquitecturas'),(pd.DataFrame(list(seed_metadata.values())),'Checkpoints'),
        (pd.DataFrame([{'campo':k,'valor':str(v)} for k,v in PROTO.items()]),'Protocolo'),
        (pd.DataFrame([{'limitacion':x} for x in REPORT['limitaciones']]),'Limitaciones')]:
        frame.to_excel(writer,sheet_name=name,index=False)
        ws=writer.sheets[name]
        ws.freeze_panes='A2'
        ws.auto_filter.ref=ws.dimensions
        from openpyxl.styles import Font, PatternFill, Alignment
        for cell in ws[1]:
            cell.fill=PatternFill('solid',fgColor='193750')
            cell.font=Font(name='Calibri',size=11,bold=True,color='FFFFFF')
            cell.alignment=Alignment(wrap_text=True,vertical='center')
        ws.row_dimensions[1].height=32
        for col in ws.columns:
            letter=col[0].column_letter
            length=max((len(str(v.value or '')) for v in list(col)[:130]),default=10)
            ws.column_dimensions[letter].width=min(38,max(13,length+2))
save_json_atomic(REPORT_FILE,REPORT)
zip_path=EXPORT/'CERDOS_23_YOLO26_Evaluacion_Resultados.zip'
files_to_zip=[EXCEL,PROTO_FILE,REPORT_FILE]
for seed in SEMILLAS:
    files_to_zip.extend([OUTPUT/f'CERDOS_23_Pred_seed_{seed}.json',OUTPUT/f'CERDOS_23_Pred_seed_{seed}_hash.json'])
for stem in ['CERDOS_23_AP_Global_3_Modelos','CERDOS_23_AP_Por_Fecha_Semillas','CERDOS_23_Conteo_Umbral_Fijo']:
    files_to_zip.extend([FIGURES/(stem+'.png'),FIGURES/(stem+'.pdf')])
assert all(p.is_file() for p in files_to_zip),'Falta algún archivo de resultados'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in files_to_zip:z.write(p,arcname=p.name)
with zipfile.ZipFile(zip_path,'r') as z:assert z.testzip() is None,'ZIP incompleto o corrupto'
print('✅ Excel:',EXCEL)
print('✅ Informe JSON:',REPORT_FILE)
print('✅ ZIP íntegro:',zip_path, '| MiB:',round(zip_path.stat().st_size/1024**2,1))
print('✅ Archivos ZIP:',len(files_to_zip))
from google.colab import files
files.download(str(zip_path))

## 10. Qué viene después

1. Envíame el ZIP **`CERDOS_23_YOLO26_Evaluacion_Resultados.zip`** para auditar las tres semillas.
2. **No** cambies el umbral de conteo según estos resultados test. Si queremos conteo comparable, debemos elegir su umbral utilizando solo **validación** en otro notebook y registrar esa elección.
3. Entrenar más semillas de YOLO11 y Mask R-CNN, verificar sus curvas y **comparar presupuestos de entrenamiento**.
4. Posteriormente armonizar calibración, geometría, conteo e inferencia para las tres arquitecturas, sin exagerar el alcance externo.

**Recuerda:** 373 imágenes test y 5 fechas no equivalen a cinco granjas ni a miles de observaciones independientes. Las diferencias observadas no deben interpretarse automáticamente como significativas.